In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

conf = SparkConf()
conf.set('spark.app.name', 'SparkML Tutorial')
conf.set('spark.master', 'local[4]')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

## Classifiaction

In [4]:
# https://www.kaggle.com/competitions/titanic/data

titanic_raw = spark.read.csv('titanic/train.csv', header = True, inferSchema = True)

titanic_raw.printSchema()
titanic_raw.show()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|   

In [ ]:
import pyspark.sql.functions as f

titanic = titanic_raw.select('Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked')\
                    .withColumn('tmp_name', f.split(f.col('Name'), ',')[1])\
                    .withColumn('Status', f.trim(f.split(f.col('tmp_name'), '\.')[0]))\
                    .drop('Name', 'tmp_name')

titanic.cache()
titanic.select('*').summary().show()

+-------+-------------------+------------------+------+------------------+------------------+-------------------+-----------------+--------+------------+
|summary|           Survived|            Pclass|   Sex|               Age|             SibSp|              Parch|             Fare|Embarked|      Status|
+-------+-------------------+------------------+------+------------------+------------------+-------------------+-----------------+--------+------------+
|  count|                891|               891|   891|               714|               891|                891|              891|     889|         891|
|   mean| 0.3838383838383838| 2.308641975308642|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|    NULL|        NULL|
| stddev|0.48659245426485753|0.8360712409770491|  NULL|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|    NULL|        NULL|
|    min|                  0|                 1|female|              0.42|  

### 1. Transformer

In [ ]:
# VectorAssembler

from pyspark.ml.feature import VectorAssembler

inputCols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch']
assembler = VectorAssembler(inputCols = inputCols, outputCol = 'vector_features', handleInvalid = 'skip')
assembler.transform(titanic).show(5, truncate = False)

+--------+------+------+----+-----+-----+-------+--------+------+----------------------+
|Survived|Pclass|Sex   |Age |SibSp|Parch|Fare   |Embarked|Status|vector_features       |
+--------+------+------+----+-----+-----+-------+--------+------+----------------------+
|0       |3     |male  |22.0|1    |0    |7.25   |S       |Mr    |[0.0,3.0,22.0,1.0,0.0]|
|1       |1     |female|38.0|1    |0    |71.2833|C       |Mrs   |[1.0,1.0,38.0,1.0,0.0]|
|1       |3     |female|26.0|0    |0    |7.925  |S       |Miss  |[1.0,3.0,26.0,0.0,0.0]|
|1       |1     |female|35.0|1    |0    |53.1   |S       |Mrs   |[1.0,1.0,35.0,1.0,0.0]|
|0       |3     |male  |35.0|0    |0    |8.05   |S       |Mr    |[0.0,3.0,35.0,0.0,0.0]|
+--------+------+------+----+-----+-----+-------+--------+------+----------------------+
only showing top 5 rows


In [ ]:
# Imputer

from pyspark.ml.feature import Imputer

age_imputer = Imputer(inputCol = 'Age', outputCol = 'Age_Imp', strategy = 'median')

model = age_imputer.fit(titanic)
model.transform(titanic).filter('Age IS NULL').show(5)

+--------+------+------+----+-----+-----+------+--------+------+-------+
|Survived|Pclass|   Sex| Age|SibSp|Parch|  Fare|Embarked|Status|Age_Imp|
+--------+------+------+----+-----+-----+------+--------+------+-------+
|       0|     3|  male|NULL|    0|    0|8.4583|       Q|    Mr|   28.0|
|       1|     2|  male|NULL|    0|    0|  13.0|       S|    Mr|   28.0|
|       1|     3|female|NULL|    0|    0| 7.225|       C|   Mrs|   28.0|
|       0|     3|  male|NULL|    0|    0| 7.225|       C|    Mr|   28.0|
|       1|     3|female|NULL|    0|    0|7.8792|       Q|  Miss|   28.0|
+--------+------+------+----+-----+-----+------+--------+------+-------+
only showing top 5 rows


In [ ]:
# Scaler

from pyspark.ml.feature import StandardScaler, MinMaxScaler

assembler = VectorAssembler(inputCols = ['Age', 'Fare'], outputCol = 'vector_feature', handleInvalid = 'skip')

standard = StandardScaler(inputCol = 'vector_feature', outputCol = 'standard_scaled')
minmax = MinMaxScaler(inputCol = 'vector_feature', outputCol = 'minmax_scaled')

vectorized = assembler.transform(titanic)

standard_model = standard.fit(vectorized)
minmax_model = minmax.fit(vectorized)

In [14]:
standard_model.transform(vectorized.select('vector_feature')).show(5, truncate = False)

+--------------+----------------------------------------+
|vector_feature|standard_scaled                         |
+--------------+----------------------------------------+
|[22.0,7.25]   |[1.5144738264626911,0.13700201550092822]|
|[38.0,71.2833]|[2.6159093366173756,1.3470283822837676] |
|[26.0,7.925]  |[1.7898327040013622,0.14975737556480773]|
|[35.0,53.1]   |[2.4093901784633722,1.0034216583585225] |
|[35.0,8.05]   |[2.4093901784633722,0.152119479280341]  |
+--------------+----------------------------------------+
only showing top 5 rows


In [15]:
minmax_model.transform(vectorized.select('vector_feature')).show(5, truncate = False)

+--------------+------------------------------------------+
|vector_feature|minmax_scaled                             |
+--------------+------------------------------------------+
|[22.0,7.25]   |[0.2711736617240513,0.014151057562208049] |
|[38.0,71.2833]|[0.4722292033174164,0.13913573538264068]  |
|[26.0,7.925]  |[0.32143754712239253,0.015468569817999833]|
|[35.0,53.1]   |[0.43453128926866047,0.10364429745562033] |
|[35.0,8.05]   |[0.43453128926866047,0.015712553569072387]|
+--------------+------------------------------------------+
only showing top 5 rows


In [ ]:
# Encoder

from pyspark.ml.feature import StringIndexer, OneHotEncoder

inputCols = ['Sex', 'Status']
outputCols = ['Sex_idx', 'Status_idx']

indexer = StringIndexer(inputCols = inputCols, outputCols = outputCols, handleInvalid = 'keep')
indexer_model = indexer.fit(titanic)

indexed = indexer_model.transform(titanic.select('Sex', 'Status'))
indexed.show(5)

+------+------+-------+----------+
|   Sex|Status|Sex_idx|Status_idx|
+------+------+-------+----------+
|  male|    Mr|    0.0|       0.0|
|female|   Mrs|    1.0|       2.0|
|female|  Miss|    1.0|       1.0|
|female|   Mrs|    1.0|       2.0|
|  male|    Mr|    0.0|       0.0|
+------+------+-------+----------+
only showing top 5 rows


In [18]:
onehot = OneHotEncoder(inputCols = outputCols, outputCols = ['Sex_OneHot', 'Status_OneHot'], handleInvalid = 'keep')

onehot_model = onehot.fit(indexed)
onehot_model.transform(indexed).show(5)

+------+------+-------+----------+-------------+--------------+
|   Sex|Status|Sex_idx|Status_idx|   Sex_OneHot| Status_OneHot|
+------+------+-------+----------+-------------+--------------+
|  male|    Mr|    0.0|       0.0|(2,[0],[1.0])|(17,[0],[1.0])|
|female|   Mrs|    1.0|       2.0|(2,[1],[1.0])|(17,[2],[1.0])|
|female|  Miss|    1.0|       1.0|(2,[1],[1.0])|(17,[1],[1.0])|
|female|   Mrs|    1.0|       2.0|(2,[1],[1.0])|(17,[2],[1.0])|
|  male|    Mr|    0.0|       0.0|(2,[0],[1.0])|(17,[0],[1.0])|
+------+------+-------+----------+-------------+--------------+
only showing top 5 rows


### 2. Estimator

In [23]:
test_raw = spark.read.csv('titanic/test.csv', header = True, inferSchema = True)
test_label = spark.read.csv('titanic/gender_submission.csv', header = True, inferSchema = True)

titanic_test = test_raw.select('PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked')\
                    .join(test_label, 'PassengerId', how = 'inner')\
                    .withColumn('tmp_name', f.split(f.col('Name'), ',')[1])\
                    .withColumn('Status', f.trim(f.split(f.col('tmp_name'), '\.')[0]))\
                    .drop('PassengerId', 'Name', 'tmp_name')

titanic_test.cache()
titanic_test.select('*').summary().show()

+-------+------------------+------+------------------+------------------+------------------+------------------+--------+-------------------+------+
|summary|            Pclass|   Sex|               Age|             SibSp|             Parch|              Fare|Embarked|           Survived|Status|
+-------+------------------+------+------------------+------------------+------------------+------------------+--------+-------------------+------+
|  count|               418|   418|               332|               418|               418|               417|     418|                418|   418|
|   mean|2.2655502392344498|  NULL|30.272590361445783|0.4473684210526316|0.3923444976076555|  35.6271884892086|    NULL|0.36363636363636365|  NULL|
| stddev|0.8418375519640503|  NULL|14.181209235624424|0.8967595611217135|0.9814288785371694|55.907576179973844|    NULL| 0.4816221409322309|  NULL|
|    min|                 1|female|              0.17|                 0|                 0|               0.0| 

In [25]:
imputer = Imputer(inputCols = ['Age', 'Fare'], outputCols = ['Age_Imp', 'Fare_Imp'], strategy = 'median')
imputer_model = imputer.fit(titanic)

assembler = VectorAssembler(inputCols = ['Pclass', 'Age_Imp', 'SibSp', 'Parch', 'Fare_Imp'], outputCol = 'features')

In [26]:
from pyspark.ml.classification import RandomForestClassifier

rf_clf = RandomForestClassifier(featuresCol = 'features',
                                labelCol = 'Survived',
                                predictionCol = 'prediction',
                                seed = 42,
                                maxDepth = 5,
                                maxBins = 32,
                                numTrees = 20)

train_data = imputer_model.transform(titanic)
train_data = assembler.transform(train_data)

rf_model = rf_clf.fit(train_data)

test_data = imputer_model.transform(titanic_test)
test_data = assembler.transform(test_data)

prediction = rf_model.transform(test_data)

In [27]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(predictionCol = 'prediction', labelCol = 'Survived', metricName = 'accuracy')
evaluator.evaluate(prediction)

0.6698564593301436

## Regression

In [32]:
# https://www.kaggle.com/datasets/sooyoungher/california-housing/data

california = spark.read.csv('california_housing.csv', header = True, inferSchema = True)

california.printSchema()
california.show()
california.select('*').summary().show()

root
 |-- MedInc: double (nullable = true)
 |-- HouseAge: double (nullable = true)
 |-- AveRooms: double (nullable = true)
 |-- AveBedrms: double (nullable = true)
 |-- Population: double (nullable = true)
 |-- AveOccup: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- MedHouseVal: double (nullable = true)

+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+
|MedInc|HouseAge|          AveRooms|         AveBedrms|Population|          AveOccup|Latitude|Longitude|MedHouseVal|
+------+--------+------------------+------------------+----------+------------------+--------+---------+-----------+
|8.3252|    41.0| 6.984126984126984|1.0238095238095237|     322.0|2.5555555555555554|   37.88|  -122.23|      4.526|
|8.3014|    21.0| 6.238137082601054|0.9718804920913884|    2401.0| 2.109841827768014|   37.86|  -122.22|      3.585|
|7.2574|    52.0| 8.288135593220339| 1.07344

In [47]:
train, test = california.randomSplit([0.8, 0.2], seed = 42)

In [51]:
inputCols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
outputCol = 'vector_features'

assembler = VectorAssembler(inputCols = inputCols, outputCol = outputCol)
vectored_train = assembler.transform(train).select('MedHouseVal', 'vector_features')
vectored_test = assembler.transform(test).select('MedHouseVal', 'vector_features')

scaler = StandardScaler(inputCol = 'vector_features', outputCol = 'scaled_vector')
scaler_model = scaler.fit(vectored_train)
scaled_train = scaler_model.transform(vectored_train).select('MedHouseVal', 'scaled_vector')
scaled_test = scaler_model.transform(vectored_test).select('MedHouseVal', 'scaled_vector')

scaled_train.show(5, truncate = False)
scaled_test.show(5, truncate = False)

+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|MedHouseVal|scaled_vector                                                                                                                                           |
+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|0.55       |[0.263364040162313,0.7933234684776896,2.712068375124473,4.271094867148665,0.09423164418035959,0.18647542098152245,16.267580753287724,-58.3848880259061] |
|0.567      |[0.263364040162313,1.1899852027165345,4.666242896447698,5.362743813241358,0.11429949432988061,0.19841033259080118,18.95930498286027,-61.59131215872318] |
|1.0        |[0.263364040162313,1.8246439774986862,2.4362524114282427,3.5021455238652153,0.17275801433065924,0.3107923683025374,16.92409885806152,-59.92816693095357]

In [56]:
from pyspark.ml.regression import RandomForestRegressor

rf_reg = RandomForestRegressor(featuresCol = 'scaled_vector',
                               labelCol = 'MedHouseVal',
                               predictionCol = 'prediction',
                               seed = 42,
                               maxDepth = 5,
                               maxBins = 32,
                               numTrees = 20)

rf_model = rf_reg.fit(scaled_train)
pred = rf_model.transform(scaled_test)

In [57]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(predictionCol = 'prediction', labelCol = 'MedHouseVal', metricName = 'rmse')
evaluator.evaluate(pred)

0.6788004567594651

## Pipeline

In [58]:
train_raw = spark.read.csv('titanic/train.csv', header = True, inferSchema = True)
test_raw = spark.read.csv('titanic/test.csv', header = True, inferSchema = True)
test_label = spark.read.csv('titanic/gender_submission.csv', header = True, inferSchema = True)

train = train_raw.na.fill({'Embarked': 'C'})\
                    .select('Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked')\
                    .withColumn('tmp_name', f.split(f.col('Name'), ',')[1])\
                    .withColumn('Status', f.trim(f.split(f.col('tmp_name'), '\.')[0]))\
                    .drop('Name', 'tmp_name')

test = test_raw.na.fill({'Embarked': 'C'})\
                    .select('PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked')\
                    .join(test_label, 'PassengerId', how = 'inner')\
                    .withColumn('tmp_name', f.split(f.col('Name'), ',')[1])\
                    .withColumn('Status', f.trim(f.split(f.col('tmp_name'), '\.')[0]))\
                    .drop('PassengerId', 'Name', 'tmp_name')

In [60]:
cat_inputCols = ['Sex', 'Embarked', 'Status']
cat_outputCols = ['Sex_Idx', 'Embarked_Idx', 'Status_Idx']
indexer = StringIndexer(inputCols = cat_inputCols, outputCols = cat_outputCols, handleInvalid = 'keep')

num_inputCols = ['Age', 'Fare']
num_outputCols = ['Age_Imp', 'Fare_Imp']
num_imputer = Imputer(inputCols = num_inputCols, outputCols = num_outputCols, strategy = 'median')
num_vector = VectorAssembler(inputCols = num_outputCols, outputCol = 'num_vector')
num_scaler = StandardScaler(inputCol = 'num_vector', outputCol = 'num_scaled')

fin_inputCols = ['Pclass', 'SibSp', 'Parch', 'num_scaled'] + cat_outputCols
fin_assembler = VectorAssembler(inputCols = fin_inputCols, outputCol = 'features')

rf_clf = RandomForestClassifier(featuresCol = 'features',
                                labelCol = 'Survived',
                                predictionCol = 'prediction',
                                seed = 42)

In [61]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages = [indexer,
                              num_imputer,
                              num_vector,
                              num_scaler,
                              fin_assembler,
                              rf_clf])

model = pipeline.fit(train)
pred = model.transform(test)

evaluator = MulticlassClassificationEvaluator(predictionCol = 'prediction', labelCol = 'Survived', metricName = 'accuracy')
evaluator.evaluate(pred)

0.937799043062201

## HyperParameter Tuning

In [65]:
from pyspark.ml.tuning import ParamGridBuilder

paramgrid = (ParamGridBuilder()
             .addGrid(rf_clf.maxDepth, [5, 6, 7, 8])
             .addGrid(rf_clf.maxBins, [32, 48, 64, 80])
             .addGrid(rf_clf.numTrees, [15, 20, 25, 30])
             .build())

In [63]:
from pyspark.ml.tuning import TrainValidationSplit

tvs = TrainValidationSplit(
    estimator = pipeline,
    estimatorParamMaps = paramgrid,
    evaluator = evaluator,
    trainRatio = 0.8
)

tvs_model = tvs.fit(train)
pred = tvs_model.transform(test)
evaluator.evaluate(pred)

0.8899521531100478

In [68]:
best_rf_model = tvs_model.bestModel.stages[-1]

print(best_rf_model.getOrDefault('maxDepth'))
print(best_rf_model.getOrDefault('maxBins'))
print(best_rf_model.getOrDefault('numTrees'))

7
32
25


In [ ]:
from pyspark.ml.tuning import CrossValidator

cv = CrossValidator(
    estimator = pipeline,
    estimatorParamMaps = paramgrid,
    evaluator = evaluator,
    numFolds = 5,
)

cv_model = cv.fit(train)
pred = cv_model.transform(test)
evaluator.evaluate(pred)

0.8947368421052632

In [69]:
best_rf_model = cv_model.bestModel.stages[-1]

print(best_rf_model.getOrDefault('maxDepth'))
print(best_rf_model.getOrDefault('maxBins'))
print(best_rf_model.getOrDefault('numTrees'))

7
48
15
